# Atom metadata pipeline — computer-networks knowledge graph

Builds and enriches a topic graph for *Computer Networking: A Top-Down Approach (8th ed.)*:

1. **Topic tree** — recursively decompose each chapter of the course markdown into atomic learning units (LLM + textbook retrieval, cached, checkpointed after every chapter).
2. **Atom metadata** — for every atom, retrieve supporting textbook chunks (pgvector search) and fill learning objective, estimated time, difficulty, type, misconceptions and source chunks (provenance-checked in code).
3. **Cross-branch prerequisites** — embed all atoms, propose cross-chapter candidate pairs (similarity + shared source chunks), one symmetric LLM verdict per pair, keep edges with confidence >= 0.7.

**Prerequisites**

- Google Drive: `MyDrive/graph/content/computer_networks.md` — course text whose first-level chapter headings look like `1: Chapter Title` (`load_roots` expects 6–10 of them).
- A PostgreSQL database with **pgvector** and a populated `book_chunks` table (produced by *chunk.ipynb*), embedding dim **1024** (`BAAI/bge-large-en-v1.5`).
- An **OpenRouter** API key (`OPENROUTER_API_KEY`); optionally set `LLM_MODEL` and `OPENROUTER_BASE_URL` as environment variables or Colab Secrets.
- The database URL goes in `DATABASE_URL` (use the `postgresql+psycopg://` scheme for psycopg 3).

**How to run:** top to bottom. Secrets are resolved from environment variables → Colab Secrets → a masked prompt, so nothing is hardcoded. Every phase caches LLM calls and checkpoints results, so interrupting and re-running is safe (already-judged pairs and annotated atoms are skipped).

**Cleaned vs the original upload:** pip installs consolidated into one cell; all secrets moved out of code (the NVIDIA test cell had a hardcoded API key — removed); the missing config globals (`DATABASE_URL`, `BASE_URL`, `LLM_MODEL`) are defined in one Configuration cell; the superseded first `call_llm` implementation and the fully commented-out Phase 5 draft were deleted (the retry-enabled client is kept); duplicated prompt/helper definitions collapsed; the three entry points that were all named `run()` (and shadowed each other) are now `run_decompose()` / `run_atom_meta()` / `run_cross_branch()`; Drive paths gathered into the Configuration cell (`ATOMS_PATH` still points at `MyDrive/Atom/checkpoints/atoms.json` as in the original — point `ATOM_BASE_DIR` at `BASE_DIR` if you prefer one folder); stale outputs and ~110 KB of unused widget state cleared.

Note: the original run's last error (`call_llm failed ... no JSON object found in output`) means the configured model returned empty/non-JSON content — pick a model that answers with plain JSON in `LLM_MODEL` and re-run; cached pairs resume automatically.


## 1. Setup — dependencies

One cell for everything the pipeline needs (quiet install, safe to re-run).


In [1]:
%pip install -q psycopg[binary] psycopg2-binary sqlalchemy python-dotenv pgvector openai sentence-transformers


## 2. Configuration — secrets & paths

All credentials and Drive paths in one place. Secrets: environment variable → Colab Secret → masked prompt.

Paths follow the original layout: the topic tree, cross edges and the shared LLM cache live under `MyDrive/graph`, and Phase 4 wrote `atoms.json` under `MyDrive/Atom`.


In [2]:
# --- Mount Google Drive ------------------------------------------------------
from google.colab import userdata
from google.colab import drive

drive.mount('/content/drive')

# --- Paths -------------------------------------------------------------------
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/graph')
COURSE_FILE = BASE_DIR / 'content' / 'ai_engineering.md'
CHECKPOINT_PATH = BASE_DIR / 'checkpoints' / 'tree.json'
CROSS_EDGES_PATH = BASE_DIR / 'checkpoints' / 'cross_edges.json'
CACHE_DIR = BASE_DIR / 'checkpoints' / 'llm_cache'  # shared by all phases
ATOMS_PATH = BASE_DIR / 'checkpoints' / 'atoms.json'

# --- Secrets & model ---------------------------------------------------------
import os
from getpass import getpass

DATABASE_URL = userdata.get('SUPABASE_DB_URL')                # PostgreSQL URL with pgvector
OPENROUTER_API_KEY = userdata.get('API_KEY')    # https://openrouter.ai/keys
BASE_URL = userdata.get('BASE_URL')
LLM_MODEL = userdata.get('LLM_MODEL')    # <-- set the OpenRouter model you want

print('Model        :', LLM_MODEL)
print('Provider   :', BASE_URL)
print('Course file  :', COURSE_FILE)
print('Tree         :', CHECKPOINT_PATH)
print('Atoms        :', ATOMS_PATH)
print('Cross edges  :', CROSS_EDGES_PATH)
print('LLM cache    :', CACHE_DIR)
# DATABASE_URL is never printed — it contains the password


Mounted at /content/drive
Model        : nvidia/nemotron-3.5-super-vl-120b-a12b
Provider   : https://integrate.api.nvidia.com/v1
Course file  : /content/drive/MyDrive/graph/content/ai_engineering.md
Tree         : /content/drive/MyDrive/graph/checkpoints/tree.json
Atoms        : /content/drive/MyDrive/graph/checkpoints/atoms.json
Cross edges  : /content/drive/MyDrive/graph/checkpoints/cross_edges.json
LLM cache    : /content/drive/MyDrive/graph/checkpoints/llm_cache


## 3. Database — engine, session, ORM model, event log

SQLAlchemy engine bound to `DATABASE_URL`, the `book_chunks` ORM model (pgvector column), and the append-only `events` logger used by `call_llm` for auditing every LLM call.


In [3]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, declarative_base

engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


In [4]:
from sqlalchemy import Column, Integer, String, Text
from pgvector.sqlalchemy import Vector

EMBEDDING_DIM = 1024  # Sync this with the embedding model you use.

class BookChunk(Base):
    __tablename__ = "book_chunks"

    id = Column(Integer, primary_key=True, index=True)
    chunk_id = Column(String, unique=True, index=True)
    source_title = Column(String, nullable=True)
    page_number = Column(Integer, nullable=True)
    content = Column(Text, nullable=False)
    token_count = Column(Integer, nullable=True)
    embedding = Column(Vector(EMBEDDING_DIM), nullable=True)


In [5]:
# src/bachelor_s_project_i_lms/db/events.py
"""Append-only event logging."""
import os
from typing import Optional
from uuid import UUID

import psycopg
from dotenv import load_dotenv
from psycopg.types.json import Jsonb

load_dotenv()


def log_event(
    type: str,
    payload: dict,
    learner_id: Optional[UUID] = None,
    graph_version: Optional[int] = None,
    model_version: Optional[str] = None,
    prompt_version: Optional[str] = None,
) -> int:
    """Insert one row into `events` and return its event_id."""
    db_url = DATABASE_URL
    if not db_url:
        raise RuntimeError("DATABASE_URL is not set in environment variables.")

    with psycopg.connect(db_url) as conn:
        row = conn.execute(
            """
            INSERT INTO events
                (type, payload, learner_id, graph_version, model_version, prompt_version)
            VALUES (%s, %s, %s, %s, %s, %s)
            RETURNING event_id
            """,
            (type, Jsonb(payload), learner_id, graph_version,
             model_version, prompt_version),
        ).fetchone()
        return row[0]


## 4. Pydantic schemas

Strict output contracts for every LLM call (decomposition, atom metadata, cross-branch verdicts) plus the question-bank and runtime diagnosis models.


In [6]:
# src/bachelor_s_project_i_lms/llm/schemas.py
from typing import Literal, Optional
from pydantic import BaseModel, ConfigDict, Field, model_validator

NodeType = Literal["concept", "practical", "test", "reading"]


# ---------- Graph building ----------

class Misconception(BaseModel):
    id: str = Field(min_length=1, description="short slug, e.g. tcp_udp_ack_confusion")
    description: str = Field(min_length=5)


class SubTopic(BaseModel):
    title: str = Field(min_length=2)
    summary: str = Field(min_length=5)
    is_atomic: bool


class DecomposeOutput(BaseModel):
    """Step 2: split a topic into 3-7 sub-topics."""
    subtopics: list[SubTopic] = Field(min_length=3, max_length=7)

    @model_validator(mode="after")
    def unique_titles(self):
        titles = [s.title.strip().lower() for s in self.subtopics]
        if len(titles) != len(set(titles)):
            raise ValueError("duplicate subtopic titles")
        return self


class LocalEdge(BaseModel):
    prerequisite: str
    dependent: str
    confidence: float = Field(ge=0, le=1)
    rationale: str = Field(min_length=5)

    @model_validator(mode="after")
    def not_self_loop(self):
        if self.prerequisite.strip().lower() == self.dependent.strip().lower():
            raise ValueError("self-loop edge")
        return self


class LocalPrereqOutput(BaseModel):
    """Step 3: prerequisite edges among the siblings only."""
    edges: list[LocalEdge] = Field(default_factory=list)


class CrossBranchVerdict(BaseModel):
    """Global pass: is node X a prerequisite of node Y?"""
    is_prerequisite: bool
    confidence: float = Field(ge=0, le=1)
    rationale: str = Field(min_length=5)


class AtomMetadata(BaseModel):
    """Metadata filled for every atomic node."""
    learning_objective: str = Field(min_length=10)
    estimated_time_min: int = Field(ge=1, le=10)
    difficulty_weight: int = Field(ge=1, le=5)
    type: NodeType
    common_misconceptions: list[Misconception] = Field(min_length=2, max_length=4)
    source_chunk_ids: list[str] = Field(min_length=1)


# ---------- Question bank ----------

class McqOption(BaseModel):
    key: Literal["A", "B", "C", "D"]
    text: str = Field(min_length=1)


class McqItem(BaseModel):
    style: Literal["conceptual", "application"]
    stem: str = Field(min_length=10)
    options: list[McqOption] = Field(min_length=4, max_length=4)
    correct_key: Literal["A", "B", "C", "D"]
    distractor_map: dict[str, str]

    @model_validator(mode="after")
    def check_consistency(self):
        keys = {o.key for o in self.options}
        if keys != {"A", "B", "C", "D"}:
            raise ValueError("options must have keys A, B, C, D exactly once")
        wrong = keys - {self.correct_key}
        if set(self.distractor_map) != wrong:
            raise ValueError("distractor_map must cover exactly the wrong options")
        if any(not misconception_id.strip() for misconception_id in self.distractor_map.values()):
            raise ValueError("each distractor requires a non-empty misconception_id")
        return self


class RubricCriterion(BaseModel):
    criterion: str = Field(min_length=3)
    max_points: int = Field(ge=1, le=3)


class OpenItem(BaseModel):
    style: Literal["conceptual", "application"]
    stem: str = Field(min_length=10)
    reference_answer: str = Field(min_length=5)
    rubric: list[RubricCriterion] = Field(min_length=2, max_length=5)


class ItemVerification(BaseModel):
    """A separate solver model answers the item; used to reject bad items."""
    solver_answer: str
    is_unambiguous: bool
    matches_key: bool


# ---------- Runtime: diagnosis and judging ----------

class ErrorDiagnosis(BaseModel):
    error_type: Literal["SLIP", "MISCONCEPTION", "PREREQ_GAP", "UNKNOWN"]
    confidence: float = Field(ge=0, le=1)
    evidence: str = Field(min_length=5)
    misconception_id: Optional[str] = None
    suspected_prereq_title: Optional[str] = None

    @model_validator(mode="after")
    def type_specific(self):
        if self.error_type == "MISCONCEPTION" and not self.misconception_id:
            raise ValueError("MISCONCEPTION requires misconception_id")
        if self.error_type == "PREREQ_GAP" and not self.suspected_prereq_title:
            raise ValueError("PREREQ_GAP requires suspected_prereq_title")
        return self


class CriterionScore(BaseModel):
    criterion: str
    points: int = Field(ge=0)
    evidence: str


class OpenAnswerJudgement(BaseModel):
    """Judge for open answers. The mapping to correct/incorrect is done in code."""
    model_config = ConfigDict(extra="forbid")

    scores: list[CriterionScore] = Field(min_length=1)
    confidence: float = Field(ge=0, le=1)


## 5. Prompts

All system prompts and message builders in one place: atom metadata (Phase 4), topic decomposition, and the symmetric cross-branch verdict (Phase 5).


In [17]:
import re

# ---------- atom metadata (Phase 4) ----------

ATOM_META_SYSTEM = """You are an expert computer-networks instructor cataloguing atomic learning units.
Given an atomic topic and excerpts from the course textbook, produce its metadata.

Rules:
- learning_objective: one measurable sentence starting with "The learner can ...".
- estimated_time_min: 1-10 (an atom is teachable in about 10 minutes).
- difficulty_weight: 1 (easy) to 5 (hard).
- type: one of "concept", "practical", "test", "reading".
- common_misconceptions: 2-4 items, each with a short snake_case id and a one-sentence description.
- source_chunk_ids: 1-3 ids of the excerpts that support this atom. Use ONLY ids from the provided excerpts — never invent one.
- Base the metadata strictly on the provided excerpts; do not invent content.
- Answer with ONLY a JSON object: {"learning_objective": str, "estimated_time_min": int, "difficulty_weight": int, "type": str, "common_misconceptions": [{"id": str, "description": str}], "source_chunk_ids": [str]}
"""


def build_atom_meta_messages(atom: dict, chunks: list[dict]) -> list[dict]:
    refs = "\n\n".join(
        f"[chunk_id={c['chunk_id']} | page={c['page']}]\n{(c['text'] or '')[:800]}" for c in chunks
    )
    valid_ids = ", ".join(str(c["chunk_id"]) for c in chunks)
    summary = (atom.get("summary") or "").strip()
    context = f"\nContext from its parent: {summary}" if summary else ""
    user = (
        f"Atomic topic: {atom['title']}{context}\n\n"
        f"Textbook excerpts for this topic:\n{refs}\n\n"
        f"Valid chunk ids (use only these): {valid_ids}\n\n"
        "Produce the metadata for this atomic topic."
    )
    return [{"role": "user", "content": user}]

#----------- topic prereqs ------------

LOCAL_PREREQ_SYSTEM = """You are an expert computer-networks instructor building a prerequisite graph.
Given a parent topic and its immediate children, decide which children are prerequisites for which other children.

Rules:
- An edge X -> Y means: to understand Y, X should be learned first.
- Only propose edges between the listed children, using their exact titles.
- Only propose an edge for a real, specific dependency — not thematic similarity or same-chapter proximity.
- A child can depend on several others; propose each dependency as its own edge.
- If the children are independent, return an empty edge list.
- Answer with ONLY a JSON object: {"edges": [{"prerequisite": str, "dependent": str, "confidence": float, "rationale": str}]}
"""


def build_local_prereq_messages(parent_title: str, children: list[dict]) -> list[dict]:
    lines = []
    for c in children:
        kind = "atomic leaf" if c["is_atomic"] else "compound (has subtopics)"
        summary = (c.get("summary") or "").strip()
        lines.append(f"- {c['title']} ({kind}): {summary}")
    user = (
        f"Parent topic: {parent_title}\n\n"
        f"Children:\n{''.join(line + chr(10) for line in lines)}\n"
        "List the prerequisite edges among these children (prerequisite -> dependent). "
        "Use exact child titles; return an empty list if none."
    )
    return [{"role": "user", "content": user}]


# ---------- topic decomposition ----------

DECOMPOSE_SYSTEM = """You are an expert computer-networks instructor building a knowledge graph.
Given a topic and excerpts from the course textbook, split the topic into 3 to 7 sub-topics.

Rules:
- A sub-topic is "atomic" ONLY if it can be taught in about 10 minutes and tested with 2-3 questions.
- Prefer fewer, well-scoped children over many vague ones. No overlap between siblings.
- Sub-topic titles must be short, specific, and unique (no synonyms of each other).
- Do not invent topics that have no support in the provided excerpts.
- Answer with ONLY a JSON object: {"subtopics": [{"title": str, "summary": str, "is_atomic": bool}, ...]}
"""


def build_decompose_messages(title: str, summary_hint: str, chunks: list[dict], depth: int) -> list[dict]:
    refs = "\n\n".join(
        f"[chunk_id={c['chunk_id']} | page={c['page']}]\n{(c['text'] or '')[:800]}" for c in chunks
    )
    hint = f"\nContext from its parent: {summary_hint}" if summary_hint else ""
    user = (
        f"Topic: {title}\n"
        f"Current depth in the tree: {depth} (max 4){hint}\n\n"
        f"Textbook excerpts for this topic:\n{refs}\n\n"
        "Split this topic into 3-7 sub-topics. Mark is_atomic=true only for leaves."
    )
    return [{"role": "user", "content": user}]


# ---------- cross-branch prerequisites (Phase 5) ----------
# Both atoms are presented symmetrically (same fields, same order, no pre-drawn
# arrow and no "earlier/later" framing) so the model judges direction from the
# content. Book position is included as plain evidence — knowing one atom is a
# chapter-1 intro and the other a chapter-5 topic is legitimate signal — but the
# prompt never suggests that page order implies dependency.

CROSS_BRANCH_SYSTEM = """You are an expert computer-networks instructor building a prerequisite graph.
Given two atomic learning units from DIFFERENT chapters of the course, decide for each direction
whether mastering one is a prerequisite for understanding the other.

Rules:
- A prerequisite means: without first mastering X, a learner cannot meaningfully understand Y.
  Merely being related, similar, or taught in the same area is NOT a prerequisite.
- Judge each direction independently. Usually at most one direction is true; often neither is.
- Use the learning objectives (what the learner must be able to do), not just the titles.
- The chapter and page numbers are evidence of depth and placement, NOT of dependency order:
  a later chapter can still be independent, and an occasional true dependency can point backward.
- confidence: 0.0-1.0, how sure you are of that direction's verdict.
- rationale: one sentence per direction.
- Answer with ONLY a JSON object:
  {"a_before_b": {"is_prerequisite": bool, "confidence": 0.0-1.0, "rationale": str},
   "b_before_a": {"is_prerequisite": bool, "confidence": 0.0-1.0, "rationale": str}}
"""


def _page_range(atom: dict) -> str:
    """'p48-55' from the atom's source chunks; empty when unparseable."""
    pages = []
    for cid in atom.get("source_chunk_ids", []):
        m = re.search(r"p(\d+)", str(cid))
        if m:
            pages.append(int(m.group(1)))
    if not pages:
        return ""
    return f"p{min(pages)}-{max(pages)}"


def _atom_block(atom: dict) -> str:
    """Symmetric presentation: identical fields in identical order for both atoms."""
    pages = _page_range(atom)
    place = f"chapter: {atom['root']}"
    if pages:
        place += f", book pages: {pages}"
    return (
        f"Title: {atom['title']}\n"
        f"Placement: {place}\n"
        f"Learning objective: {atom.get('learning_objective', '')}"
    )


def build_cross_branch_messages(atom_a: dict, atom_b: dict) -> list[dict]:
    user = (
        f"Atom A\n{_atom_block(atom_a)}\n\n"
        f"Atom B\n{_atom_block(atom_b)}\n\n"
        "Q1: Is A a prerequisite for B? (a_before_b)\n"
        "Q2: Is B a prerequisite for A? (b_before_a)"
    )
    return [{"role": "user", "content": user}]


## 6. LLM client — `call_llm` with retry, auditing and JSON parsing

Unified OpenRouter call layer: retries transient API errors (timeouts, 429, 5xx) with exponential backoff, retries schema-validation failures by showing the model its bad output, extracts the outermost JSON object, and logs every attempt to the `events` table.


In [8]:
# =============================================================================
# llm/client.py  —  unified LLM call layer (retry API errors + validation)
# =============================================================================
"""Unified LLM call layer with retry, error logging, and JSON schema parsing via OpenRouter."""

import json
import logging
import os
import re
import time
from typing import Any, Callable, Optional, Type, TypeVar

from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError

load_dotenv()

OPENROUTER_BASE_URL = BASE_URL
LLM_MAX_TOKENS = 8192
DEFAULT_PROMPT_VERSION = "v1"

# Per-call HTTP timeout (seconds). Reasoning models can take well over a minute
# to finish thinking + writing the answer, so keep this generous.
LLM_TIMEOUT_S = 120.0

logger = logging.getLogger(__name__)
T = TypeVar("T", bound=BaseModel)

GenerateFn = Callable[[list[dict], Optional[str]], tuple[str, dict]]

# Reasoning/thinking models (R1, Qwen3-thinking, nemotron, o1/o3/o4, ...) spend
# their `max_tokens` budget on hidden reasoning BEFORE writing any visible text:
# with a small cap that yields an EMPTY response (finish_reason="length").
# The unified OpenRouter `reasoning` parameter turns thinking off for models
# that support it; gateways that reject the unknown field fall back automatically
# (one 400, then the parameter is disabled for the rest of the session).
DISABLE_REASONING = True
_reasoning_param_ok = True


# ----------------------------- retry helpers -----------------------------

def _is_retryable(err: Exception) -> bool:
    """True for transient errors worth retrying (timeouts, connection drops, 429, 5xx)."""
    try:
        from openai import (
            APIConnectionError,
            APIStatusError,
            APITimeoutError,
            RateLimitError,
        )
    except ImportError:
        return False

    if isinstance(err, (APITimeoutError, APIConnectionError, RateLimitError)):
        return True
    if isinstance(err, APIStatusError):
        return err.status_code in (429, 500, 502, 503, 504)
    return False


def _backoff_seconds(attempt: int) -> float:
    """1s, 2s, 4s, 8s, capped at 16s."""
    return min(2 ** attempt, 16)


# ----------------------------- generate -----------------------------

def _openrouter_generate(
    messages: list[dict],
    system: Optional[str],
    max_tokens: Optional[int] = None,
) -> tuple[str, dict]:
    """Send one chat completion. SDK-level retries are disabled — call_llm owns retry policy."""
    global _reasoning_param_ok

    from openai import APIStatusError, OpenAI

    api_key = os.getenv("OPENROUTER_API_KEY", globals().get("OPENROUTER_API_KEY"))
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set in environment or .env file.")

    client = OpenAI(
        base_url=BASE_URL,
        api_key=api_key,
        timeout=LLM_TIMEOUT_S,
        max_retries=0,  # IMPORTANT: our loop retries; the SDK must not
    )

    formatted_messages = []
    if system:
        formatted_messages.append({"role": "system", "content": system})
    formatted_messages.extend(messages)

    model = os.getenv("LLM_MODEL", globals().get("LLM_MODEL"))

    def _send(extra: dict):
        return client.chat.completions.create(
            model=model,
            messages=formatted_messages,
            temperature=0.0,
            max_tokens=max_tokens or LLM_MAX_TOKENS,
            extra_body=extra,
        )

    extra_body = {}
    if DISABLE_REASONING and _reasoning_param_ok:
        extra_body["reasoning"] = {"enabled": False, "exclude": True}

    try:
        response = _send(extra_body)
    except APIStatusError as err:
        if extra_body and err.status_code == 400:
            # Gateway doesn't know the unified reasoning param: drop it for good.
            _reasoning_param_ok = False
            logger.warning("gateway rejected the `reasoning` parameter — retrying without it")
            response = _send({})
        else:
            raise

    choice = response.choices[0]
    text = choice.message.content or ""
    if not text.strip() and choice.finish_reason == "stop":
        # Last resort: a thinking gateway that answers inside reasoning_content.
        rc = getattr(choice.message, "reasoning_content", None) or ""
        if rc.strip():
            text = rc

    detail = getattr(response.usage, "completion_tokens_detail", None) if response.usage else None
    usage = {
        "input_tokens": getattr(response.usage, "prompt_tokens", 0) if response.usage else 0,
        "output_tokens": getattr(response.usage, "completion_tokens", 0) if response.usage else 0,
        # reasoning_tokens > 0 with empty text = thinking ate the whole budget
        "reasoning_tokens": getattr(detail, "reasoning_tokens", None),
        # finish_reason so truncated ("length") responses show up in logs
        "finish_reason": getattr(choice, "finish_reason", None),
    }
    return text, usage


# ----------------------------- JSON extraction -----------------------------

def _scan_from(block: str, start: int):
    """String-aware scan for a balanced {...} starting at block[start] == '{'.

    Returns (candidate_str, is_complete, resume_pos).
    For truncated output, returns a best-effort repaired string (dangling
    quote/braces closed) so pydantic can report a precise error downstream.
    """
    depth, in_str, esc = 0, False, False
    for i in range(start, len(block)):
        ch = block[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        elif ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return block[start : i + 1], True, i + 1
    # Truncated: close the dangling string and braces.
    snippet = block[start:].rstrip()
    if in_str:
        snippet += '"'
    if depth > 0:
        snippet += "}" * depth
    return snippet, False, None


def _extract_json(raw: str) -> str:
    """Best-effort extraction of a JSON object from raw LLM output.

    Handles markdown fences, prose around the object, stray braces in prose,
    and truncated output. Raises ValueError only when there are no braces at
    all (e.g. empty or pure-prose response).
    """
    if not raw or not raw.strip():
        raise ValueError("no JSON object found in output (empty response)")

    # Candidate blocks: fenced chunks first, then the raw text itself.
    blocks = [
        m.group(1)
        for m in re.finditer(r"```(?:json)?\s*(.*?)\s*```", raw, flags=re.DOTALL | re.IGNORECASE)
    ]
    blocks.append(raw)

    fallback: Optional[str] = None
    for block in blocks:
        pos = 0
        while True:
            start = block.find("{", pos)
            if start == -1:
                break
            candidate, complete, resume = _scan_from(block, start)
            if complete:
                try:
                    json.loads(candidate)
                    return candidate  # clean, complete, parseable
                except json.JSONDecodeError:
                    if fallback is None:
                        fallback = candidate
                    pos = resume  # keep scanning past this malformed object
            else:
                # Truncated output: return the repaired string; pydantic will
                # raise a precise error (missing field) instead of a confusing EOF.
                if fallback is None:
                    fallback = candidate
                break

    if fallback is not None:
        return fallback
    raise ValueError("no JSON object found in output")


# ----------------------------- call_llm -----------------------------

def call_llm(
    messages: list[dict],
    response_model: Type[T],
    system: Optional[str] = None,
    learner_id: Optional[Any] = None,
    prompt_version: str = DEFAULT_PROMPT_VERSION,
    generate: Optional[GenerateFn] = None,
    max_retries: int = 3,
    max_tokens: Optional[int] = None,
) -> T:
    """
    Call the model, validate against a Pydantic schema, log events.

    Retries BOTH transient API errors (timeouts, 429, 5xx) and schema
    validation failures. Backs off exponentially between API retries.
    """
    gen = generate or _openrouter_generate
    model_name = os.getenv("LLM_MODEL", globals().get("LLM_MODEL"))
    history_messages = list(messages)
    last_error: Optional[Exception] = None

    # Escalate the output budget on each retry (1x, 2x, 3x ... capped at
    # LLM_MAX_TOKENS) so a truncated JSON gets more room on the next attempt
    # instead of failing identically.
    def _call_gen(msgs, attempt):
        if generate is None:
            attempt_tokens = (
                min(max_tokens * (attempt + 1), LLM_MAX_TOKENS)
                if max_tokens is not None
                else None
            )
            return gen(msgs, system, max_tokens=attempt_tokens)
        return gen(msgs, system)

    for attempt in range(max_retries + 1):
        try:
            raw_text, usage = _call_gen(history_messages, attempt)
        except Exception as api_err:
            last_error = api_err
            logger.error("LLM API call failed (attempt %d): %s", attempt, api_err)
            try:
                log_event(
                    type="llm_error",
                    payload={
                        "error_type": "api_error",
                        "error_message": str(api_err),
                        "attempt": attempt,
                        "retryable": _is_retryable(api_err),
                    },
                    learner_id=learner_id,
                    model_version=model_name,
                    prompt_version=prompt_version,
                )
            except Exception:
                pass

            if _is_retryable(api_err) and attempt < max_retries:
                time.sleep(_backoff_seconds(attempt))
                continue  # retry the same messages
            raise api_err  # non-retryable or out of retries

        try:
            clean_json = _extract_json(raw_text)
            parsed_data = response_model.model_validate_json(clean_json)

            try:
                log_event(
                    type="llm_call",
                    payload={
                        "status": "success",
                        "attempt": attempt,
                        "usage": usage,
                        "output": parsed_data.model_dump(),
                    },
                    learner_id=learner_id,
                    model_version=model_name,
                    prompt_version=prompt_version,
                )
            except Exception as db_err:
                logger.warning("Could not log success event to DB: %s", db_err)

            return parsed_data

        except (ValueError, ValidationError) as err:
            last_error = err
            # Log the raw head + finish_reason + reasoning tokens to the console
            # so failures are diagnosable without opening the DB.
            logger.warning(
                "Attempt %d validation failed: %s | finish_reason=%s | reasoning_tokens=%s | raw head: %r",
                attempt,
                err,
                (usage or {}).get("finish_reason"),
                (usage or {}).get("reasoning_tokens"),
                raw_text[:200],
            )

            try:
                log_event(
                    type="llm_validation_error",
                    payload={
                        "status": "validation_failed",
                        "attempt": attempt,
                        "error": str(err),
                        "raw_output": raw_text,
                        "usage": usage,
                    },
                    learner_id=learner_id,
                    model_version=model_name,
                    prompt_version=prompt_version,
                )
            except Exception as db_err:
                logger.warning("Could not log validation error event to DB: %s", db_err)

            if attempt < max_retries:
                # Tailor the corrective message to the failure mode.
                if (usage or {}).get("finish_reason") == "length":
                    retry_hint = (
                        "Your previous response was cut off before the JSON was complete. "
                        "Reply again with the COMPLETE JSON object, no markdown fences, "
                        "no prose, and keep any 'rationale' strings under 20 words."
                    )
                else:
                    retry_hint = (
                        f"Your previous response caused a validation error: {err}. "
                        f"Please provide ONLY valid JSON matching the required schema."
                    )
                history_messages.append({"role": "assistant", "content": raw_text})
                history_messages.append({"role": "user", "content": retry_hint})

    raise RuntimeError(
        f"call_llm failed after {max_retries + 1} attempts. Last error: {last_error}"
    )


## 7. Embeddings & vector search

`BAAI/bge-large-en-v1.5` (1024 dims, normalized — must match *chunk.ipynb*), a lazy singleton loader, and `search()` for quick sanity checks against `book_chunks`. The same `embed()` is reused by the pipeline for retrieval and Phase 5 similarity.


In [9]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"  # 1024 dimensions, must match chunk.ipynb

_model = None  # lazy singleton: load once per process

def _get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    return _model

def embed(text: str) -> list[float]:
    """Same embedding config as in chunk.ipynb (bge-large-en-v1.5, normalized)."""
    vec = _get_model().encode([text], normalize_embeddings=True)[0]
    return [float(x) for x in vec]

def search(query: str, k: int = 3):
    """Quick sanity check: nearest book chunks for a free-text query."""
    query_vec = embed(query)
    db = SessionLocal()
    try:
        results = (
            db.query(BookChunk)
            .order_by(BookChunk.embedding.cosine_distance(query_vec))
            .limit(k)
            .all()
        )
        print(f"\n🔎 Query: {query}")
        for rank, c in enumerate(results, start=1):
            preview = (c.content or "")[:160].replace("\n", " ")
            print(f"\n[{rank}] chunk_id={c.chunk_id} | page={c.page_number} | source={c.source_title}")
            print(f"    {preview}...")

        return results
    finally:
        db.close()

# Example (first call downloads the ~1.3 GB model):
# search("a sentence related to one of your chapter titles")


## 8. Phase 1–3 — topic tree (decompose)

Parses the chapter roots from the course markdown, retrieves textbook chunks per topic (vector search), and recursively expands each topic until its children are atomic or `MAX_DEPTH` is reached. Results are cached per `(title, depth)` and checkpointed to `tree.json` after every chapter, so an interrupted run resumes without paying twice.


In [10]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from pathlib import Path
from typing import Optional

# Paths (BASE_DIR, COURSE_FILE, CHECKPOINT_PATH, CACHE_DIR) are defined in the
# Configuration cell above.

MAX_DEPTH = 4
TOP_K_CHUNKS = 8


def _norm(title: str) -> str:
    """Normalize a title for cache keys and dedup checks."""
    s = unicodedata.normalize("NFKD", title).lower().strip()
    return re.sub(r"\s+", " ", s)


def _tid(*parts: str) -> str:
    """Short deterministic temp id (NOT the final node_id; that comes from freeze)."""
    return "t_" + hashlib.sha256("\x1f".join(parts).encode()).hexdigest()[:10]


# ----------------------------- step 1: roots -----------------------------

def load_roots(path: Path = COURSE_FILE) -> list[str]:
    """Parse first-level chapter titles from the course markdown (one root each)."""
    roots: list[str] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        m = re.match(r"^\s*\d+\s*[::]\s*(.+)$", line)
        if m:
            roots.append(m.group(1).strip())
    if not (6 <= len(roots) <= 10):
        raise ValueError(f"expected 6-10 root chapters, got {len(roots)} — fix the course file first")
    return roots


# ----------------------------- chunk retrieval -----------------------------

def fetch_chunks(query: str, k: int = TOP_K_CHUNKS) -> list[dict]:
    """Vector search book_chunks for a topic title. Returns [{chunk_id, page, text}]."""
    # local import: keeps sentence_transformers (a heavy ML dep) out of module import time

    qvec = embed(query)
    db = SessionLocal()
    try:
        rows = (
            db.query(BookChunk)
            .order_by(BookChunk.embedding.cosine_distance(qvec))
            .limit(k)
            .all()
        )
        return [{"chunk_id": c.chunk_id, "page": c.page_number, "text": c.content} for c in rows]
    finally:
        db.close()


# ----------------------------- LLM + cache -----------------------------

def _decompose_cache_key(title: str, depth: int) -> Path:
    h = hashlib.sha256(f"{_norm(title)}\x1f{depth}".encode()).hexdigest()
    return CACHE_DIR / f"{h}.json"


def decompose_topic(title: str, summary_hint: str, depth: int, generate=None) -> DecomposeOutput:
    """Split one topic into 3-7 subtopics. Cached: identical (title, depth) never re-calls the LLM."""
    key = _decompose_cache_key(title, depth)
    if key.exists():
        return DecomposeOutput.model_validate_json(key.read_text(encoding="utf-8"))

    chunks = fetch_chunks(title)
    messages = build_decompose_messages(title=title, summary_hint=summary_hint, chunks=chunks, depth=depth)
    out = call_llm(messages=messages, response_model=DecomposeOutput, system=DECOMPOSE_SYSTEM, generate=generate)

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    key.write_text(out.model_dump_json(indent=2), encoding="utf-8")
    return out


# ----------------------------- recursion -----------------------------

def _new_node(title: str, summary: str, parent_tid: Optional[str], depth: int, root: str) -> dict:
    return {
        "temp_id": _tid(root, title),
        "title": title,
        "summary": summary,
        "depth": depth,
        "root": root,               # syllabus chapter this subtree belongs to (coverage check later)
        "parent_temp_id": parent_tid,
        "is_atomic": False,
        "needs_review": False,
        "children": [],
    }


def expand(node: dict, generate=None) -> dict:
    """Recursively expand `node` in place until leaves are atomic or MAX_DEPTH is hit."""
    title, depth = node["title"], node["depth"]
    print(f"{'  ' * depth}- {title} (depth {depth})")

    if depth >= MAX_DEPTH:
        # Depth cap reached: force it to be an atom, flag for human review (step 7).
        node["is_atomic"] = True
        node["needs_review"] = True
        return node

    out = decompose_topic(title, node.get("summary", ""), depth, generate=generate)

    seen = {_norm(title)}  # avoid child with same title as parent
    for st in out.subtopics:
        t = _norm(st.title)
        if t in seen:
            continue
        seen.add(t)
        child = _new_node(st.title, st.summary, node["temp_id"], depth + 1, node["root"])
        if st.is_atomic:
            child["is_atomic"] = True
        else:
            expand(child, generate=generate)
        node["children"].append(child)

    # If every child ended up merged/duplicate (rare), treat this node itself as an atom.
    if not node["children"]:
        node["is_atomic"] = True
        node["needs_review"] = True
    return node


# ----------------------------- checkpoint -----------------------------

def save_checkpoint(tree: list[dict], path: Path = CHECKPOINT_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(tree, ensure_ascii=False, indent=2), encoding="utf-8")


def load_checkpoint(path: Path = CHECKPOINT_PATH) -> Optional[list[dict]]:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def iter_atoms(tree: list[dict]):
    """Yield every leaf/atom node in the forest, in document order."""
    for node in tree:
        if node["is_atomic"]:
            yield node
        else:
            yield from iter_atoms(node["children"])


# ----------------------------- entry point -----------------------------

def run_decompose(generate=None, resume: bool = True) -> list[dict]:
    """Build the full topic tree for all roots. Safe to interrupt: results are cached."""
    roots = load_roots(COURSE_FILE)
    tree: list[dict] = []
    done = {n["temp_id"] for n in (load_checkpoint() or [])} if resume else set()

    for r in roots:
        root_node = _new_node(r, summary="", parent_tid=None, depth=0, root=r)
        if root_node["temp_id"] in done:
            print(f"= {r} (already in checkpoint, skipping)")
            continue
        expand(root_node, generate=generate)
        tree.append(root_node)
        save_checkpoint(tree)  # save after every root so a crash loses at most one chapter
    return tree


## 9. Phase 4 - Local prerequisites

After each decomposition, make only one call_llm("local_prereq", ..., LocalPrereqOutput) among the same children.

One subtle point: when two children are not yet atomic and A becomes a prerequisite of B, the actual edge must be between leaves. The proposed rule: the "output" leaves of A (leaves in A's subtree that nothing depends on) connect to the "input" leaves of B (leaves in B's subtree that have no internal prerequisite). Do this transformation in code, not with the LLM.

In [20]:
"""Phase 3: local prerequisite edges among siblings, expanded to leaf-to-leaf edges.

For every non-leaf node we make one ``local_prereq`` LLM call asking for prerequisite
edges between its children. An edge between two *atomic* children is kept as-is.
When a non-atomic child A is a prerequisite of a non-atomic child B, the edge is
rewritten in code (never by the LLM) between leaves: the "output" leaves of A
(sinks of A's internal DAG — nothing in A depends on them) connect to the "input"
leaves of B (sources of B's internal DAG — no internal prerequisite feeds them).

All state lives next to the decomposition checkpoints: raw LLM outputs are cached
per (parent, children) so an interrupted run never pays twice, and the final
leaf-level edges are written to ``checkpoints/edges.json`` with temporary IDs
(stable node_ids are only created during freeze).
"""
from __future__ import annotations

import hashlib
import json
from typing import Optional

EDGES_PATH = BASE_DIR / "checkpoints" / "edges.json"
CACHE_DIR = BASE_DIR / "checkpoints" / "llm_cache"  # shared with decompose


# ----------------------------- LLM + cache -----------------------------

def _cache_key(node: dict) -> Path:
    """Cache key covers the parent title AND its children: a re-decomposed parent
    (different children) must not reuse a stale verdict."""
    child_titles = sorted(_norm(c["title"]) for c in node["children"])
    h = hashlib.sha256("\x1f".join([_norm(node["title"]), str(node["depth"]), *child_titles]).encode()).hexdigest()
    return CACHE_DIR / f"prereq_{h}.json"


def local_prereqs(node: dict, generate=None) -> LocalPrereqOutput:
    """One LLM call: prerequisite edges among the children of `node`. Cached."""
    key = _cache_key(node)
    if key.exists():
        return LocalPrereqOutput.model_validate_json(key.read_text(encoding="utf-8"))

    messages = build_local_prereq_messages(node["title"], node["children"])
    out = call_llm(messages=messages, response_model=LocalPrereqOutput, system=LOCAL_PREREQ_SYSTEM, generate=generate)

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    key.write_text(out.model_dump_json(indent=2), encoding="utf-8")
    return out


# ----------------------------- leaf helpers -----------------------------

def _subtree_leaves(node: dict) -> list[dict]:
    if node["is_atomic"] or not node["children"]:
        return [node]
    out: list[dict] = []
    for c in node["children"]:
        out.extend(_subtree_leaves(c))
    return out


def _sinks(node: dict, edges: list[dict]) -> list[dict]:
    """Leaves of node's subtree that nothing (within the subtree) depends on:
    the terminal outcomes of learning this topic."""
    has_outgoing = {e["from_temp_id"] for e in edges}
    return [l for l in _subtree_leaves(node) if l["temp_id"] not in has_outgoing]


def _sources(node: dict, edges: list[dict]) -> list[dict]:
    """Leaves of node's subtree with no internal prerequisite: entry points."""
    has_incoming = {e["to_temp_id"] for e in edges}
    return [l for l in _subtree_leaves(node) if l["temp_id"] not in has_incoming]


# ----------------------------- edge records -----------------------------

def _dedupe(edges: list[dict]) -> list[dict]:
    """Same (from, to) pair may arise from several sibling edges: keep the best."""
    best: dict[tuple[str, str], dict] = {}
    for e in edges:
        k = (e["from_temp_id"], e["to_temp_id"])
        if k not in best or e["confidence"] > best[k]["confidence"]:
            best[k] = e
    return sorted(best.values(), key=lambda e: (e["from_temp_id"], e["to_temp_id"]))


# ----------------------------- recursion -----------------------------

def leaf_edges(node: dict, generate=None) -> list[dict]:
    """All leaf-level prerequisite edges inside `node`'s subtree.

    Post-order: each child's internal leaf edges are computed first, so the
    sibling-edge transformation can find sinks/sources inside their subtrees.
    """
    if not node["children"]:
        return []

    per_child = {c["temp_id"]: leaf_edges(c, generate=generate) for c in node["children"]}
    edges = [e for es in per_child.values() for e in es]

    by_norm = {_norm(c["title"]): c for c in node["children"]}
    for e in local_prereqs(node, generate=generate).edges:
        a = by_norm.get(_norm(e.prerequisite))
        b = by_norm.get(_norm(e.dependent))
        if a is None or b is None or a is b:
            continue  # model referenced something that is not a child (or itself) — drop
        a_edges = per_child[a["temp_id"]]
        b_edges = per_child[b["temp_id"]]
        transformed = not (a["is_atomic"] and b["is_atomic"])
        for src in _sinks(a, a_edges):
            for dst in _sources(b, b_edges):
                rationale = e.rationale
                if transformed:
                    rationale += f" [expanded from sibling edge '{a['title']}' -> '{b['title']}']"
                edges.append(
                    {
                        "from_temp_id": src["temp_id"],
                        "to_temp_id": dst["temp_id"],
                        "confidence": e.confidence,
                        "rationale": rationale,
                        "origin": "local",
                    }
                )
    return _dedupe(edges)


# ----------------------------- checkpoint -----------------------------

def save_edges(edges: list[dict], path: Path = EDGES_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(edges, ensure_ascii=False, indent=2), encoding="utf-8")


def load_edges(path: Path = EDGES_PATH) -> Optional[list[dict]]:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


# ----------------------------- entry point -----------------------------

def run_prereqs(generate=None, tree_path: Path = CHECKPOINT_PATH, edges_path: Path = EDGES_PATH) -> list[dict]:
    """Compute leaf-level local edges for the whole checkpointed tree.

    Safe to interrupt: every LLM call is cached, and edges are saved after each
    root so a crash loses at most one chapter.
    """
    tree = load_checkpoint(tree_path)
    if tree is None:
        raise RuntimeError("no tree checkpoint found — run decompose first")

    all_edges: list[dict] = []
    for root in tree:
        all_edges.extend(leaf_edges(root, generate=generate))
        save_edges(all_edges, edges_path)
        print(f"= {root['title']}: {len(all_edges)} leaf edges so far")
    return all_edges


## 10. Phase 5 — atom metadata

For each atom: retrieve the relevant textbook chunks (vector search), one `call_llm(..., AtomMetadata)` call, attach the result to the node. Provenance is strict — any `chunk_id` the model cites that is not among the retrieved database chunks is rejected in code; if nothing survives, the atom keeps no metadata and is flagged `needs_review` for the human pass. Raw outputs are cached per `(atom, chunk set)` and the enriched tree is checkpointed after every root.


In [11]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Optional

# ATOMS_PATH and CACHE_DIR are defined in the Configuration cell above.


# ----------------------------- LLM + cache -----------------------------

def _atom_meta_cache_key(atom: dict, chunks: list[dict]) -> Path:
    """Cache key covers the atom title AND the retrieved chunk set: re-embedding or a
    changed retrieval must not reuse a stale metadata verdict."""
    chunk_ids = sorted(str(c["chunk_id"]) for c in chunks)
    h = hashlib.sha256("\x1f".join([_norm(atom["title"]), *chunk_ids]).encode()).hexdigest()
    return CACHE_DIR / f"atommeta_{h}.json"


def atom_meta(atom: dict, generate=None, fetch=None) -> Optional[AtomMetadata]:
    """Metadata for one atom, cached. Returns None (and flags the atom) when the model
    cites no valid database chunk: metadata without provenance is worthless."""
    chunks = (fetch or fetch_chunks)(atom["title"])
    key = _atom_meta_cache_key(atom, chunks)
    if key.exists():
        out = AtomMetadata.model_validate_json(key.read_text(encoding="utf-8"))
    else:
        messages = build_atom_meta_messages(atom, chunks)
        out = call_llm(messages=messages, response_model=AtomMetadata, system=ATOM_META_SYSTEM, generate=generate)
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        key.write_text(out.model_dump_json(indent=2), encoding="utf-8")

    # Provenance check in code: the model may only cite chunks we actually retrieved.
    valid_ids = {str(c["chunk_id"]) for c in chunks}
    out.source_chunk_ids = [cid for cid in out.source_chunk_ids if cid in valid_ids]
    if not out.source_chunk_ids:
        atom["needs_review"] = True
        return None

    atom["learning_objective"] = out.learning_objective
    atom["estimated_time_min"] = out.estimated_time_min
    atom["difficulty_weight"] = out.difficulty_weight
    atom["type"] = out.type
    atom["common_misconceptions"] = [m.model_dump() for m in out.common_misconceptions]
    atom["source_chunk_ids"] = out.source_chunk_ids
    return out


# ----------------------------- checkpoint -----------------------------

def save_atoms(atoms: list[dict], path: Path = ATOMS_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(atoms, ensure_ascii=False, indent=2), encoding="utf-8")


def load_atoms(path: Path = ATOMS_PATH) -> Optional[list[dict]]:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


# ----------------------------- entry point -----------------------------

def run_atom_meta(generate=None, fetch=None, tree_path: Path = CHECKPOINT_PATH, atoms_path: Path = ATOMS_PATH) -> tuple[list[dict], int]:
    """Fill metadata for every atom in the checkpointed tree.

    Already-annotated atoms are skipped, results are checkpointed after every root, and
    every LLM call is cached — safe to interrupt and resume.
    Returns (tree, n_flagged) where n_flagged counts atoms left for human review.
    """
    tree = load_checkpoint(tree_path)
    if tree is None:
        raise RuntimeError("no tree checkpoint found — run decompose first")

    n_flagged = 0
    done_atoms: list[dict] = []
    for root in tree:
        root_atoms = list(iter_atoms([root]))
        for atom in root_atoms:
            if atom.get("learning_objective"):
                continue  # already annotated (previous interrupted run)
            if atom_meta(atom, generate=generate, fetch=fetch) is None:
                n_flagged += 1
        save_checkpoint(tree, tree_path)
        done_atoms.extend({k: a[k] for k in a if k != "children"} for a in root_atoms)
        save_atoms(done_atoms, atoms_path)
        print(f"= {root['title']}: {sum(1 for a in root_atoms if a.get('learning_objective'))}/{len(root_atoms)} atoms annotated")
    return tree, n_flagged


## 11. Phase 6 — cross-branch prerequisite edges

The local pass (Phase 3) only ordered siblings, so the 8 chapter graphs are disconnected islands. This phase stitches them together:

1. Embed every atom as `title + learning objective` (same bge model as the chunks; one batch, no LLM).
2. Candidates: for each atom, the top-k most similar atoms *from other roots*, plus any cross-root pair sharing a `source_chunk_id` (the "same concept at two depths" case similarity can miss).
3. One LLM call per unique pair judges BOTH directions symmetrically; book position is evidence in the prompt, never a rule.
4. Code verifies: only edges with confidence >= 0.7 are kept with `origin = "cross_branch"`; a kept edge pointing from a later book page to an earlier one is flagged `backward` for review.

Every LLM call is cached per pair and edges are saved every 25 judged pairs, so interrupted runs resume without repeating work.


In [12]:
# =============================================================================
# cross_branch.py  —  Phase 5: cross-branch prerequisite edges
# =============================================================================
"""Phase 5: cross-branch prerequisite edges between the chapter islands.

The local pass (Phase 3) only ordered siblings, so the 8 chapter graphs are
disconnected. This phase stitches them together:

1. Embed every atom as ``title + learning objective`` (same bge model as the
   chunks; done in one batch, no LLM).
2. Candidates: for each atom, the top-k most similar atoms *from other roots*,
   plus — as a second, free generator — any cross-root pair sharing a
   ``source_chunk_id`` (two atoms grounded in the same textbook passage are the
   "same concept at two depths" case similarity can miss).
3. One LLM call per unique pair judges BOTH directions symmetrically; book
   position is evidence in the prompt, never a rule.
4. Code verifies: titles must exist, verdicts must be well-formed, same-root
   and self pairs are dropped defensively, and only edges with confidence >=
   0.7 are kept with ``origin = "cross_branch"``. A kept edge that points from
   a later book page to an earlier one is flagged ``backward`` for review.

Every LLM call is cached per pair so an interrupted run never pays twice.
Results land in ``checkpoints/cross_edges.json``; merging into ``edges.json``
happens only when the full pass completes cleanly.
"""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path
from typing import Optional

from pydantic import BaseModel, Field

# --- from earlier cells ------------------------------------------------------
# CROSS_EDGES_PATH / CACHE_DIR / ATOMS_PATH: Configuration cell
# CROSS_BRANCH_SYSTEM / build_cross_branch_messages: Prompts cell
# call_llm: LLM client cell   |   embed / load_atoms: earlier cells
# ----------------------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.7
TOP_K = 8  # similar candidates per atom from other roots

# Reasoning models spend their thinking budget INSIDE max_tokens, so a small
# cap yields an empty/truncated answer (finish_reason="length"). 4096 leaves
# room for thinking + the verdict JSON; the retry loop escalates if needed.
CROSS_BRANCH_MAX_TOKENS = 4096


# ----------------------------- verdict models -----------------------------

class DirectionalVerdict(BaseModel):
    """One direction of a cross-branch pair judgment."""
    is_prerequisite: bool = Field(
        ...,
        description=(
            "True if the 'from' atom must be learned before the 'to' atom; "
            "False if no prerequisite relationship holds in that direction."
        ),
    )
    confidence: float = Field(
        ...,
        ge=0.0,
        le=1.0,
        description="Confidence in the prerequisite judgment.",
    )
    rationale: str = Field(
        ...,
        description=(
            "One-sentence justification grounded in titles, learning "
            "objectives, and book position."
        ),
    )


class CrossBranchPairVerdict(BaseModel):
    """Symmetric verdict for a pair: both directions judged in one LLM call."""
    a_before_b: DirectionalVerdict
    b_before_a: DirectionalVerdict


# ----------------------------- book position -----------------------------

def _min_page(atom: dict) -> Optional[int]:
    """Earliest book page cited by the atom, or None. Chunk ids look like
    'chunk_p48_52', so the first number after 'p' is the chunk's start page."""
    pages = []
    for cid in atom.get("source_chunk_ids", []):
        m = re.search(r"p(\d+)", str(cid))
        if m:
            pages.append(int(m.group(1)))
    return min(pages) if pages else None


# ----------------------------- similarity -----------------------------

def _cosine(u: list[float], v: list[float]) -> float:
    """Dot product of two normalized vectors (bge embeddings are normalized,
    so this is cosine similarity). Pure Python: 139 atoms is a tiny workload."""
    return sum(a * b for a, b in zip(u, v))


def _norm_title(title: str) -> str:
    return " ".join(title.strip().lower().split())


def _ordered_pair(x: str, y: str) -> tuple[str, str]:
    return (x, y) if x <= y else (y, x)


def _proposals_for_atom(
    a: dict,
    atoms: list[dict],
    vectors: dict[str, list[float]],
    top_k: int,
) -> list[tuple[str, str]]:
    """The pairs one atom proposes: its top_k nearest atoms from other roots."""
    va = vectors.get(a["temp_id"])
    if va is None:
        return []
    scored = []
    for b in atoms:
        if b["root"] == a["root"] or b["temp_id"] == a["temp_id"]:
            continue
        vb = vectors.get(b["temp_id"])
        if vb is None:
            continue
        scored.append((_cosine(va, vb), b["temp_id"]))
    # similarity desc, tie-break by id for determinism
    scored.sort(key=lambda t: (-t[0], t[1]))
    return [_ordered_pair(a["temp_id"], b_id) for _, b_id in scored[:top_k]]


def candidates(
    atoms: list[dict],
    vectors: dict[str, list[float]],
    top_k: int = TOP_K,
) -> list[tuple[str, str]]:
    """Unique cross-root candidate pairs (temp_id_a, temp_id_b), a < b.

    Two generators, unioned:
    - similarity: each atom proposes its top-k nearest atoms from other roots
      (the union is bidirectional — an atom can be proposed by others more
      often than top_k, which is fine: the cap is per atom, per spec);
    - chunk overlap: cross-root pairs sharing at least one source_chunk_id —
      grounded in the same passage, which similarity ranking can miss.
    """
    pairs: set[tuple[str, str]] = set()
    for a in atoms:
        pairs.update(_proposals_for_atom(a, atoms, vectors, top_k))

    by_chunk: dict[str, list[dict]] = {}
    for a in atoms:
        for cid in a.get("source_chunk_ids", []):
            by_chunk.setdefault(str(cid), []).append(a)
    for atoms_sharing in by_chunk.values():
        for i in range(len(atoms_sharing)):
            for j in range(i + 1, len(atoms_sharing)):
                x, y = atoms_sharing[i], atoms_sharing[j]
                if x["root"] != y["root"]:
                    pairs.add(_ordered_pair(x["temp_id"], y["temp_id"]))

    return sorted(pairs)


# ----------------------------- LLM + cache -----------------------------

def _cross_branch_cache_key(atom_a: dict, atom_b: dict) -> Path:
    """Cache key covers both titles AND objectives: a re-run of Phase 4 that
    changes an objective must not reuse a stale verdict."""
    parts = [
        _norm_title(atom_a["title"]),
        atom_a.get("learning_objective", ""),
        _norm_title(atom_b["title"]),
        atom_b.get("learning_objective", ""),
    ]
    h = hashlib.sha256("\x1f".join(parts).encode()).hexdigest()
    return CACHE_DIR / f"crossbranch_{h}.json"


def judge_pair(atom_a: dict, atom_b: dict, generate=None) -> CrossBranchPairVerdict:
    """One LLM call: both directions for the pair. Cached."""
    key = _cross_branch_cache_key(atom_a, atom_b)
    if key.exists():
        return CrossBranchPairVerdict.model_validate_json(key.read_text(encoding="utf-8"))

    messages = build_cross_branch_messages(atom_a, atom_b)
    out = call_llm(
        messages=messages,
        response_model=CrossBranchPairVerdict,
        system=CROSS_BRANCH_SYSTEM,
        generate=generate,
        max_retries=3,
        max_tokens=CROSS_BRANCH_MAX_TOKENS,
    )

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    key.write_text(out.model_dump_json(indent=2), encoding="utf-8")
    return out


# ----------------------------- edge records -----------------------------

def _edge(from_atom: dict, to_atom: dict, confidence: float, rationale: str) -> dict:
    edge = {
        "from_temp_id": from_atom["temp_id"],
        "to_temp_id": to_atom["temp_id"],
        "confidence": confidence,
        "rationale": rationale,
        "origin": "cross_branch",
    }
    # A kept edge pointing from a later book page to an earlier one is rare and
    # surprising — exactly what a human should glance at in Phase 7. If either
    # atom lacks a parseable page (e.g. front-matter with no chunk ids), we
    # simply skip the flag rather than guess a direction.
    p_from, p_to = _min_page(from_atom), _min_page(to_atom)
    if p_from is not None and p_to is not None and p_from > p_to:
        edge["backward"] = True
    return edge


def verdicts_to_edges(
    atom_a: dict,
    atom_b: dict,
    verdict: CrossBranchPairVerdict,
) -> list[dict]:
    """Turn one pair verdict into 0, 1 or 2 edges, threshold applied in code."""
    edges = []
    if (
        verdict.a_before_b.is_prerequisite
        and verdict.a_before_b.confidence >= CONFIDENCE_THRESHOLD
    ):
        edges.append(
            _edge(
                atom_a,
                atom_b,
                verdict.a_before_b.confidence,
                verdict.a_before_b.rationale,
            )
        )
    if (
        verdict.b_before_a.is_prerequisite
        and verdict.b_before_a.confidence >= CONFIDENCE_THRESHOLD
    ):
        edges.append(
            _edge(
                atom_b,
                atom_a,
                verdict.b_before_a.confidence,
                verdict.b_before_a.rationale,
            )
        )
    return edges


# ----------------------------- reporting -----------------------------

def confidence_histogram(
    verdicts: dict[tuple[str, str], CrossBranchPairVerdict],
) -> dict[str, int]:
    """Distribution of every directional confidence, so the 0.7 threshold can
    be checked against the real data before Phase 6 bakes the edges in."""
    buckets = {"0.0-0.3": 0, "0.3-0.5": 0, "0.5-0.7": 0, "0.7-0.9": 0, "0.9-1.0": 0}
    for v in verdicts.values():
        for d in (v.a_before_b, v.b_before_a):
            c = d.confidence
            if c < 0.3:
                buckets["0.0-0.3"] += 1
            elif c < 0.5:
                buckets["0.3-0.5"] += 1
            elif c < 0.7:
                buckets["0.5-0.7"] += 1
            elif c < 0.9:
                buckets["0.7-0.9"] += 1
            else:
                buckets["0.9-1.0"] += 1
    return buckets


# ----------------------------- checkpoint -----------------------------

def save_cross_edges(edges: list[dict], path: Path = CROSS_EDGES_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(edges, ensure_ascii=False, indent=2), encoding="utf-8")


def load_cross_edges(path: Path = CROSS_EDGES_PATH) -> Optional[list[dict]]:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


# ----------------------------- entry point -----------------------------

def run_cross_branch(
    generate=None,
    embed_fn=None,
    atoms_path=None,
    edges_path: Path = CROSS_EDGES_PATH,
    top_k: int = TOP_K,
) -> tuple[list[dict], dict[str, int]]:
    """Compute cross-branch edges for all atoms in the checkpointed catalog.

    Safe to interrupt: every LLM call is cached, edges are saved every 25
    judged pairs, and already-emitted edges are loaded on restart so the same
    pair is never appended twice.
    """

    atoms = load_atoms(atoms_path or ATOMS_PATH)
    if atoms is None:
        raise RuntimeError("no atoms.json found — run atom_meta first")

    by_id = {a["temp_id"]: a for a in atoms}

    embedder = embed_fn or embed
    vectors = {
        a["temp_id"]: embedder(f"{a['title']}. {a.get('learning_objective', '')}")
        for a in atoms
    }

    pairs = candidates(atoms, vectors, top_k=top_k)
    print(f"= {len(pairs)} unique candidate pairs (top_k={top_k} + chunk overlap)")

    # Resume: reload already-emitted edges and the set of pairs already judged.
    existing_edges = load_cross_edges(edges_path) or []
    edges: list[dict] = list(existing_edges)
    judged_pairs: set[tuple[str, str]] = set()
    for e in existing_edges:
        judged_pairs.add(_ordered_pair(e["from_temp_id"], e["to_temp_id"]))
    if judged_pairs:
        print(f"= resuming: {len(judged_pairs)} pairs already judged in {edges_path.name}")

    remaining = [p for p in pairs if p not in judged_pairs]
    print(f"= {len(remaining)} pairs left to judge")

    verdicts: dict[tuple[str, str], CrossBranchPairVerdict] = {}

    n_failed = 0
    for i, (a_id, b_id) in enumerate(remaining, start=1):
        try:
            verdict = judge_pair(by_id[a_id], by_id[b_id], generate=generate)
        except Exception as err:
            n_failed += 1  # not cached — a rerun will retry this pair
            print(f"! pair {i} failed ({by_id[a_id]['title']!r} <-> {by_id[b_id]['title']!r}): {err}")
            continue
        verdicts[(a_id, b_id)] = verdict
        new_edges = verdicts_to_edges(by_id[a_id], by_id[b_id], verdict)
        edges.extend(new_edges)

        if i % 25 == 0 or i == len(remaining):
            save_cross_edges(edges, edges_path)
            print(f"= judged {i}/{len(remaining)} pairs, {len(edges)} edges total")
    if n_failed:
        print(f"= NOTE: {n_failed} pair(s) failed all attempts — rerun run_cross_branch() later to retry them")

    # Histogram over all pairs (fresh + resumed-from-cache).
    all_verdicts: dict[tuple[str, str], CrossBranchPairVerdict] = dict(verdicts)
    for a_id, b_id in judged_pairs:
        if (a_id, b_id) not in all_verdicts:
            all_verdicts[(a_id, b_id)] = judge_pair(
                by_id[a_id], by_id[b_id], generate=generate
            )  # cache hit: no LLM call

    hist = confidence_histogram(all_verdicts)
    print("= confidence histogram:", json.dumps(hist))

    n_backward = sum(1 for e in edges if e.get("backward"))
    if n_backward:
        print(
            f"= NOTE: {n_backward} kept edge(s) point backward in book order "
            f"— review in Phase 7"
        )

    return edges, hist


In [13]:
import json, re

def extract_json_object(text):
    """Extract first complete JSON object; handles ```json fences, prose, truncation."""
    if not text or not text.strip():
        return None
    blocks = [m.group(1) for m in re.finditer(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL)]
    blocks.append(text)

    def _loads(s):
        try:
            obj = json.loads(s)
            return obj if isinstance(obj, dict) else None
        except json.JSONDecodeError:
            return None

    for block in blocks:
        obj = _loads(block.strip())
        if obj is not None:
            return obj
        start = block.find("{")
        if start < 0:
            continue
        depth, in_str, esc, end = 0, False, False, -1
        for i in range(start, len(block)):
            ch = block[i]
            if in_str:
                if esc: esc = False
                elif ch == "\\": esc = True
                elif ch == '"': in_str = False
            elif ch == '"': in_str = True
            elif ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        if end >= 0:
            obj = _loads(block[start:end + 1])
            if obj is not None:
                return obj
            continue
        snippet = block[start:].rstrip()
        if in_str: snippet += '"'
        snippet += "}" * depth
        obj = _loads(snippet)
        if obj is not None:
            return obj
    return None

## 12. Run the pipeline

Run each phase in order. All of them are interrupt-safe (LLM cache + checkpoints), so re-running a cell only does the remaining work. Expect the decompose and cross-branch phases to make hundreds of LLM calls — hours, not minutes.


In [14]:
# Phase 1-3: build/expand the topic tree from the course chapters.
tree = run_decompose()


- Introduction to Building AI Applications with Foundation Models (depth 0)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

- Understanding Foundation Models (depth 0)
  - Foundation Model Training Process (depth 1)
    - Pre-training (depth 2)
    - Supervised Finetuning (SFT) (depth 2)
    - Preference Finetuning (depth 2)
    - Training Design Decisions (depth 2)
      - Training Data Selection and Quality (depth 3)
      - Model Architecture and Scaling (depth 3)
        - Transformer Block Architecture (depth 4)
        - Model Capacity and Parameter Scaling (depth 4)
        - Training Bottlenecks and Resource Constraints (depth 4)
        - Inference Optimization Techniques (depth 4)
      - Post-Training Finetuning Strategies (depth 3)
      - Hyperparameter Optimization (depth 3)
      - Training Efficiency and Cost Management (depth 3)
  - Foundation Model Evaluation Challenges (depth 1)
  - Selecting Foundation Models for Applications (depth 1)
  - Adapting Foundation Models (depth 1)
- Evaluation Methodology (depth 0)
  - Evaluation Criteria Definition (depth 1)
  - Evaluation Methods and Data S

In [21]:
# Phase 4: prereqs.
prereqs = run_prereqs()

= Introduction to Building AI Applications with Foundation Models: 2 leaf edges so far
= Understanding Foundation Models: 66 leaf edges so far
= Evaluation Methodology: 110 leaf edges so far
= Evaluate AI Systems: 130 leaf edges so far
= Prompt Engineering: 134 leaf edges so far
= RAG and Agents: 149 leaf edges so far
= Finetuning: 156 leaf edges so far
= Dataset Engineering: 187 leaf edges so far
= Inference Optimization: 193 leaf edges so far
= AI Engineering Architecture and User Feedback: 193 leaf edges so far


In [15]:
# Phase 5: annotate every atom. n_flagged = atoms left for human review.
tree, n_flagged = run_atom_meta()
print(f"atoms flagged for review: {n_flagged}")


= Introduction to Building AI Applications with Foundation Models: 3/3 atoms annotated
= Understanding Foundation Models: 42/42 atoms annotated
= Evaluation Methodology: 26/26 atoms annotated
= Evaluate AI Systems: 12/12 atoms annotated
= Prompt Engineering: 4/4 atoms annotated
= RAG and Agents: 10/10 atoms annotated
= Finetuning: 5/5 atoms annotated
= Dataset Engineering: 18/18 atoms annotated
= Inference Optimization: 4/4 atoms annotated
= AI Engineering Architecture and User Feedback: 4/4 atoms annotated
atoms flagged for review: 0


In [16]:
# Phase 6: cross-branch prerequisite edges between the chapter islands.
edges, hist = run_cross_branch()


= 764 unique candidate pairs (top_k=8 + chunk overlap)
= 764 pairs left to judge
= judged 25/764 pairs, 11 edges total
= judged 50/764 pairs, 20 edges total
= judged 75/764 pairs, 28 edges total
= judged 100/764 pairs, 32 edges total
= judged 125/764 pairs, 34 edges total
= judged 150/764 pairs, 45 edges total
= judged 175/764 pairs, 50 edges total
= judged 200/764 pairs, 52 edges total
= judged 225/764 pairs, 56 edges total
= judged 250/764 pairs, 69 edges total
= judged 275/764 pairs, 74 edges total
= judged 300/764 pairs, 76 edges total
= judged 325/764 pairs, 85 edges total
= judged 350/764 pairs, 93 edges total
= judged 375/764 pairs, 98 edges total
= judged 400/764 pairs, 100 edges total
= judged 425/764 pairs, 106 edges total
= judged 450/764 pairs, 110 edges total
= judged 475/764 pairs, 112 edges total
= judged 500/764 pairs, 118 edges total


  Invalid JSON: expected value at line 1 column 75 [type=json_invalid, input_value='{"a_before_b": {"is_prer...resource constraints.}}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid | finish_reason=stop | reasoning_tokens=None | raw head: '{"a_before_b": {"is_prerequisite": false, "confidence": 0.9, "rationale": Deciding whether to finetune does not require knowing specific finetuning frameworks or tactics; one can evaluate data, hardwa'
  Invalid JSON: expected value at line 1 column 75 [type=json_invalid, input_value='{"a_before_b": {"is_prer...ource constraints.}}"}}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid | finish_reason=stop | reasoning_tokens=None | raw head: '{"a_before_b": {"is_prerequisite": false, "confidence": 0.9, "rationale": Deciding whether to finetune does not require knowing specific finetuning frameworks or tactics; one can evaluate data, hardwa'
  Invalid 

! pair 502 failed ('When to Finetune' <-> 'Finetuning Frameworks and Tactics'): call_llm failed after 4 attempts. Last error: 1 validation error for CrossBranchPairVerdict
  Invalid JSON: expected value at line 1 column 75 [type=json_invalid, input_value='{"a_before_b": {"is_prer...ource constraints.}}"}}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid
= judged 525/764 pairs, 127 edges total
= judged 550/764 pairs, 128 edges total
= judged 575/764 pairs, 134 edges total
= judged 600/764 pairs, 137 edges total
= judged 625/764 pairs, 145 edges total
= judged 650/764 pairs, 155 edges total
= judged 675/764 pairs, 159 edges total
= judged 700/764 pairs, 166 edges total
= judged 725/764 pairs, 175 edges total
= judged 750/764 pairs, 181 edges total
= judged 764/764 pairs, 187 edges total
= NOTE: 1 pair(s) failed all attempts — rerun run_cross_branch() later to retry them
= confidence histogram: {"0.0-0.3": 0, "0.3-0.5": 0, "0.5-0.7": 3, "